# 🩸 Anemia-Screen — Pipeline Walkthrough

Step-by-step view of what the pipeline does to a **single** conjunctiva image:
`Raw → Cropped → Normalized → Quality Gate → Inference → Grad-CAM → WHO Triage`.

> Run `python -m scripts.make_sample_images` first so `data/sample/` exists.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # make 'app' importable from notebooks/
import json, base64
import cv2
import numpy as np
import matplotlib.pyplot as plt
from app.schemas import PatientMetadata
from app.pipeline import preprocessing, quality_gate
from app.pipeline.orchestrator import analyze_image
print('imports OK')

In [ ]:
# Pick a sample image + its metadata sidecar
img_path = '../data/sample/pale_pregnant.png'
meta = json.load(open('../data/sample/pale_pregnant.json'))
patient = PatientMetadata(**meta)
bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
plt.figure(figsize=(4,3)); plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)); plt.title('1) Raw input'); plt.axis('off'); plt.show()
print('metadata:', meta)

In [ ]:
# 2) Quality Gate
q = quality_gate.assess_quality(bgr)
print('passed:', q.passed)
print(json.dumps(q.model_dump(), indent=2))

In [ ]:
# 3) Preprocessing: auto-crop + colour normalize
pre = preprocessing.preprocess(bgr)
fig, ax = plt.subplots(1, 2, figsize=(8,3))
ax[0].imshow(cv2.cvtColor(pre['cropped_bgr'], cv2.COLOR_BGR2RGB)); ax[0].set_title('Cropped ROI'); ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(pre['normalized_bgr'], cv2.COLOR_BGR2RGB)); ax[1].set_title('Normalized'); ax[1].axis('off')
plt.show()

In [ ]:
# 4+5+6) Full pipeline: inference + Grad-CAM + WHO triage
report = analyze_image(bgr, patient, with_gradcam=True)
rd = report.model_dump()
print('STATUS:', rd['status'])
print('VISION:', json.dumps(rd['vision'], indent=2))
print('TRIAGE:', json.dumps(rd['triage'], indent=2))
print('WARNINGS:', rd['warnings'])

In [ ]:
# Show the Grad-CAM heatmap (where the model looked)
b64 = rd.get('gradcam_png_base64')
if b64:
    arr = np.frombuffer(base64.b64decode(b64), np.uint8)
    heat = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    plt.figure(figsize=(4,4)); plt.imshow(cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)); plt.title('Grad-CAM heatmap'); plt.axis('off'); plt.show()
else:
    print('No Grad-CAM produced.')

## Macro demo (batch)
Run in a terminal to process the whole sample folder into a CSV:
```bash
python -m scripts.batch_process --mode local --input data/sample --out outputs/triage.csv
```